<a href="https://colab.research.google.com/github/reemaalzurayqi-gif/Bayan-nlp-reema/blob/main/03_text_classification_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# اليوم الثاني — مختبر 3A: تصنيف النصوص
## Day 2 — Lab 3A: Text Classification

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار:** Core → Explore → Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/main/notebooks/03_text_classification.ipynb)

> نبني Baseline أولًا، نثبت عدم تسرب المجموعات، ثم ننفذ خطوة ضبط فعلية لـDistilmBERT متعدد اللغات.
>
> Build a baseline, prove group isolation, then perform a real fine-tuning step on multilingual DistilBERT.

**علامة النجاح:** `DAY2_NOTEBOOK3_CORE=PASS`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## قواعد المختبر

- البيانات اصطناعية ولا تحتوي حالات أو أشخاصًا حقيقيين.
- GPU في Colab Free غير مضمون؛ CPU fallback يجمد المشفر ويدرب task head.
- لا نرفع model weights أو cache إلى GitHub.
- نتائج هذه العينة الصغيرة تسمى `MEASURED_SMOKE`، وليست أداءً إنتاجيًا.
- نفذ **Runtime → Run all**. لا تتجاوز خلية فاشلة.

In [ ]:
# تثبيت نسخ اليوم الثاني فقط عند الحاجة
import importlib.metadata
import importlib.util
import subprocess
import sys

REQUIRED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "scikit-learn": "1.9.0",
}
needs_install = []
for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != expected:
        needs_install.append(f"{distribution}=={expected}")

if needs_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needs_install])

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is required. Open this notebook in Google Colab.")

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

Python: 3.13.15
Environment ready / البيئة جاهزة


In [ ]:
import csv
import io
import json
import math
import os
import random
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


## 1) تحميل البيانات وفحصها

يحاول الدفتر قراءة ملف الدورة من GitHub. إذا تعذر، يستخدم عينة اصطناعية مدمجة حتى لا يتوقف Core بسبب رابط البيانات. النموذج نفسه يحتاج تنزيلًا أول مرة.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main/data/sample/bayan_day2_classification.csv"
FALLBACK_ROWS = [{"example_id":"F-001","group_id":"DG-A","split":"train","language":"ar","text":"تعذر تسجيل الدخول إلى البوابة","topic":"digital_service","sentiment":"negative"},{"example_id":"F-002","group_id":"DG-A","split":"train","language":"en","text":"I cannot sign in to the portal","topic":"digital_service","sentiment":"negative"},{"example_id":"F-003","group_id":"DG-B","split":"train","language":"ar","text":"الخدمة الإلكترونية سريعة وواضحة","topic":"digital_service","sentiment":"positive"},{"example_id":"F-004","group_id":"DG-B","split":"train","language":"en","text":"The online service is clear and fast","topic":"digital_service","sentiment":"positive"},{"example_id":"F-005","group_id":"PG-A","split":"train","language":"ar","text":"أحتاج معرفة حالة طلب التصريح","topic":"permit","sentiment":"neutral"},{"example_id":"F-006","group_id":"PG-A","split":"train","language":"en","text":"I need the status of my permit request","topic":"permit","sentiment":"neutral"},{"example_id":"F-007","group_id":"PG-B","split":"train","language":"ar","text":"تمت الموافقة على التصريح اليوم","topic":"permit","sentiment":"positive"},{"example_id":"F-008","group_id":"PG-B","split":"train","language":"en","text":"The permit was approved today","topic":"permit","sentiment":"positive"},{"example_id":"F-009","group_id":"HG-A","split":"train","language":"ar","text":"تأخر موعد العيادة هذا الصباح","topic":"health","sentiment":"negative"},{"example_id":"F-010","group_id":"HG-A","split":"train","language":"en","text":"My clinic appointment was delayed","topic":"health","sentiment":"negative"},{"example_id":"F-011","group_id":"HG-B","split":"train","language":"ar","text":"كانت خدمة العيادة ممتازة","topic":"health","sentiment":"positive"},{"example_id":"F-012","group_id":"HG-B","split":"train","language":"en","text":"The clinic service was excellent","topic":"health","sentiment":"positive"},{"example_id":"F-013","group_id":"TG-A","split":"train","language":"ar","text":"الحافلة لم تصل في الوقت المحدد","topic":"transport","sentiment":"negative"},{"example_id":"F-014","group_id":"TG-A","split":"train","language":"en","text":"The bus did not arrive on time","topic":"transport","sentiment":"negative"},{"example_id":"F-015","group_id":"TG-B","split":"train","language":"ar","text":"كانت الرحلة مريحة ومنظمة","topic":"transport","sentiment":"positive"},{"example_id":"F-016","group_id":"TG-B","split":"train","language":"en","text":"The trip was comfortable and organised","topic":"transport","sentiment":"positive"},{"example_id":"F-017","group_id":"DG-V","split":"validation","language":"ar","text":"لم يصل رمز التحقق الرقمي","topic":"digital_service","sentiment":"negative"},{"example_id":"F-018","group_id":"PG-V","split":"validation","language":"en","text":"How can I renew the permit","topic":"permit","sentiment":"neutral"},{"example_id":"F-019","group_id":"HG-V","split":"validation","language":"ar","text":"أحتاج إعادة جدولة الموعد الصحي","topic":"health","sentiment":"neutral"},{"example_id":"F-020","group_id":"TG-V","split":"validation","language":"en","text":"The bus route has changed","topic":"transport","sentiment":"neutral"},{"example_id":"F-021","group_id":"DG-T","split":"test","language":"en","text":"The verification code did not arrive","topic":"digital_service","sentiment":"negative"},{"example_id":"F-022","group_id":"PG-T","split":"test","language":"ar","text":"تأخر إصدار التصريح المطلوب","topic":"permit","sentiment":"negative"},{"example_id":"F-023","group_id":"HG-T","split":"test","language":"en","text":"The clinic appointment was cancelled","topic":"health","sentiment":"negative"},{"example_id":"F-024","group_id":"TG-T","split":"test","language":"ar","text":"توقفت الحافلة قبل المحطة","topic":"transport","sentiment":"negative"}]

try:
    with urllib.request.urlopen(DATA_URL, timeout=20) as response:
        text = response.read().decode("utf-8")
    rows = list(csv.DictReader(io.StringIO(text)))
    DATA_SOURCE = "github_course_file"
except Exception as exc:
    rows = FALLBACK_ROWS
    DATA_SOURCE = f"embedded_fallback:{type(exc).__name__}"

print("Data source:", DATA_SOURCE)
print("Rows:", len(rows))
print("Topics:", Counter(row["topic"] for row in rows))
assert len(rows) >= 24
assert {"ar", "en"} <= {row["language"] for row in rows}

Data source: github_course_file
Rows: 40
Topics: Counter({'digital_service': 10, 'permit': 10, 'health': 10, 'transport': 10})


### عقد رأسي topic وsentiment / Two independent classification heads

يحتوي ملف البيانات على `topic` و`sentiment`. تشغيل المرجع المحفوظ يقيس رأس **topic** كي يبقى smoke محدودًا، لكنه لا يُعد دليلًا لرأس sentiment. في مشروع بيان أعد استخدام split المجمّد نفسه لتدريب رأس sentiment مستقل وlabel map مستقلة، ثم قيّم الرأسين كلًا بمقياسه. تحافظ نسخة fallback كذلك على العمودين حتى لا يتغير العقد عند تعطل GitHub Raw.

In [ ]:
def validate_splits(rows):
    required = {"train", "validation", "test"}
    group_owner = {}
    labels_by_split = {name: set() for name in required}
    counts = Counter()
    for row in rows:
        split, group, label = row["split"], row["group_id"], row["topic"]
        if split not in required:
            raise ValueError(f"Unknown split: {split}")
        previous = group_owner.setdefault(group, split)
        if previous != split:
            raise ValueError(f"Group leakage: {group}")
        labels_by_split[split].add(label)
        counts[split] += 1
    all_labels = set().union(*labels_by_split.values())
    for split in required:
        if labels_by_split[split] != all_labels:
            raise ValueError(f"Missing label in {split}")
    return {
        "rows": dict(counts),
        "groups": len(group_owner),
        "labels": sorted(all_labels),
        "group_overlap": 0,
    }

split_report = validate_splits(rows)
print(json.dumps(split_report, ensure_ascii=False, indent=2))
assert split_report["group_overlap"] == 0
print("Split contract=PASS")

{
  "rows": {
    "train": 24,
    "validation": 8,
    "test": 8
  },
  "groups": 20,
  "labels": [
    "digital_service",
    "health",
    "permit",
    "transport"
  ],
  "group_overlap": 0
}
Split contract=PASS


**المتوقع:** كل split يحتوي الفئات الأربع، و`group_overlap` يساوي صفرًا.

## 2) TF-IDF Baseline

نضبط الـbaseline على Train ونقرأ Validation. لا نستخدم Test حتى خلية القياس النهائي بعد تثبيت المسار.

In [ ]:
train_rows = [row for row in rows if row["split"] == "train"]
validation_rows = [row for row in rows if row["split"] == "validation"]
test_rows = [row for row in rows if row["split"] == "test"]
LABELS = sorted({row["topic"] for row in rows})
label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1),
    LinearSVC(random_state=SEED),
)
baseline.fit(
    [row["text"] for row in train_rows],
    [row["topic"] for row in train_rows],
)
baseline_val_pred = baseline.predict([row["text"] for row in validation_rows])
baseline_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    baseline_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
print("Baseline validation macro-F1 (MEASURED_SMOKE):", round(baseline_val_f1, 4))
assert 0.0 <= baseline_val_f1 <= 1.0
print("Baseline=PASS")

Baseline validation macro-F1 (MEASURED_SMOKE): 0.6667
Baseline=PASS


In [ ]:
baseline_errors = [
    {
        "example_id": row["example_id"],
        "language": row["language"],
        "text": row["text"],
        "true_topic": row["topic"],
        "predicted_topic": prediction,
    }
    for row, prediction in zip(validation_rows, baseline_val_pred)
    if row["topic"] != prediction
]
print("Baseline validation errors:", len(baseline_errors))
print(json.dumps(baseline_errors[0], ensure_ascii=False, indent=2))
assert baseline_errors

Baseline validation errors: 2
{
  "example_id": "D-007",
  "language": "ar",
  "text": "لم يصل رمز التحقق للخدمة الرقمية",
  "true_topic": "digital_service",
  "predicted_topic": "transport"
}


### Baseline Error Analysis
The TF-IDF baseline misclassified 2 out of 8 validation examples.  
For example, the Arabic text "لم يصل رمز التحقق للخدمة الرقمية" was correctly labeled as `digital_service`, but the model predicted `transport`.

In [ ]:
import re
import unicodedata

from sklearn.pipeline import FeatureUnion, make_pipeline

def normalize_conservative(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("ـ", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text


improved_baseline = make_pipeline(
    FeatureUnion([
        (
            "word_tfidf",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=1,
            ),
        ),
        (
            "char_tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=1,
            ),
        ),
    ]),
    LinearSVC(random_state=SEED),
)

improved_baseline.fit(
    [normalize_conservative(row["text"]) for row in train_rows],
    [row["topic"] for row in train_rows],
)

improved_val_pred = improved_baseline.predict(
    [normalize_conservative(row["text"]) for row in validation_rows]
)

improved_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    improved_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)

improved_errors = [
    {
        "example_id": row["example_id"],
        "language": row["language"],
        "text": row["text"],
        "true_topic": row["topic"],
        "predicted_topic": prediction,
    }
    for row, prediction in zip(validation_rows, improved_val_pred)
    if row["topic"] != prediction
]
print("Original baseline validation macro-F1:", round(baseline_val_f1, 4))
print("Improved baseline validation macro-F1:", round(improved_val_f1, 4))
print("Original validation errors:", len(baseline_errors))
print("Improved validation errors:", len(improved_errors))

Original baseline validation macro-F1: 0.6667
Improved baseline validation macro-F1: 0.6667
Original validation errors: 2
Improved validation errors: 2


### Measured Extension Result

A conservative text-normalisation profile and combined word/character TF-IDF features were evaluated using the same group-isolated validation split.

The validation Macro-F1 remained 0.6667, with 2 errors out of 8 validation examples. Therefore, this extension was not adopted as an improvement.

The Transformer remains the selected model because it achieved a validation Macro-F1 of 1.0, compared with 0.6667 for the TF-IDF baseline. The unchanged result also shows the limitation of drawing strong conclusions from a tiny validation set.

In [ ]:
candidate_configs = [
    {
        "name": "char_wb_2_5_C0.5",
        "analyzer": "char_wb",
        "ngram_range": (2, 5),
        "C": 0.5,
    },
    {
        "name": "char_wb_2_5_C1",
        "analyzer": "char_wb",
        "ngram_range": (2, 5),
        "C": 1.0,
    },
    {
        "name": "char_wb_3_6_C1",
        "analyzer": "char_wb",
        "ngram_range": (3, 6),
        "C": 1.0,
    },
    {
        "name": "char_wb_3_5_C2",
        "analyzer": "char_wb",
        "ngram_range": (3, 5),
        "C": 2.0,
    },
    {
        "name": "char_2_5_C1",
        "analyzer": "char",
        "ngram_range": (2, 5),
        "C": 1.0,
    },
]

candidate_results = []

train_texts = [
    normalize_conservative(row["text"])
    for row in train_rows
]

validation_texts = [
    normalize_conservative(row["text"])
    for row in validation_rows
]

train_labels = [row["topic"] for row in train_rows]
validation_labels = [row["topic"] for row in validation_rows]

for config in candidate_configs:
    model = make_pipeline(
        TfidfVectorizer(
            analyzer=config["analyzer"],
            ngram_range=config["ngram_range"],
            min_df=1,
        ),
        LinearSVC(
            C=config["C"],
            random_state=SEED,
        ),
    )
    model.fit(train_texts, train_labels)
    predictions = model.predict(validation_texts)

    macro_f1 = f1_score(
        validation_labels,
        predictions,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )
    error_count = sum(
        truth != prediction
        for truth, prediction in zip(validation_labels, predictions)
    )
    candidate_results.append({
        "name": config["name"],
        "macro_f1": macro_f1,
        "errors": error_count,
        "model": model,
        "predictions": predictions,
    })

for result in candidate_results:
    print(
        result["name"],
        "| Macro-F1:", round(result["macro_f1"], 4),
        "| Errors:", result["errors"],
    )
best_candidate = max(
    candidate_results,
    key=lambda result: result["macro_f1"],
)
print("\nBest candidate:", best_candidate["name"])
print("Best validation Macro-F1:", round(best_candidate["macro_f1"], 4))
print("Best validation errors:", best_candidate["errors"])

char_wb_2_5_C0.5 | Macro-F1: 0.6667 | Errors: 2
char_wb_2_5_C1 | Macro-F1: 0.6667 | Errors: 2
char_wb_3_6_C1 | Macro-F1: 0.6667 | Errors: 2
char_wb_3_5_C2 | Macro-F1: 0.6667 | Errors: 2
char_2_5_C1 | Macro-F1: 0.65 | Errors: 2

Best candidate: char_wb_2_5_C0.5
Best validation Macro-F1: 0.6667
Best validation errors: 2


### Measured Baseline Extension

Five character-level TF-IDF configurations were evaluated using the same
group-isolated validation split.
Four candidates matched the original baseline with a validation Macro-F1 of 0.6667 and 2 errors out of 8 examples,while one candidate achieved a Macro-F1 of 0.6500 with 2 errors.

Because no measurable improvement was observed and the validation set is small,the original TF-IDF baseline was retained. The frozen test set was
not used to select or tune these configurations.

In [ ]:
baseline_per_language = {}

for language in sorted({row["language"] for row in validation_rows}):
    indices = [
        index
        for index, row in enumerate(validation_rows)
        if row["language"] == language
    ]

    language_truth = [
        validation_rows[index]["topic"]
        for index in indices
    ]

    language_predictions = [
        baseline_val_pred[index]
        for index in indices
    ]

    baseline_per_language[language] = {
        "examples": len(indices),
        "macro_f1": round(
            f1_score(
                language_truth,
                language_predictions,
                labels=LABELS,
                average="macro",
                zero_division=0,
            ),
            4,
        ),
        "accuracy": round(
            accuracy_score(language_truth, language_predictions),
            4,
        ),
        "errors": sum(
            truth != prediction
            for truth, prediction in zip(
                language_truth,
                language_predictions,
            )
        ),
    }
print(json.dumps(
    baseline_per_language,
    ensure_ascii=False,
    indent=2,
))

{
  "ar": {
    "examples": 4,
    "macro_f1": 0.6667,
    "accuracy": 0.75,
    "errors": 1
  },
  "en": {
    "examples": 4,
    "macro_f1": 0.6667,
    "accuracy": 0.75,
    "errors": 1
  }
}


## 3) تنزيل الـcheckpoint وتجهيز النموذج

النموذج عام ومجاني، ولا يحتاج Hugging Face token. عند ظهور warning بأن classification head جديدة فهذا متوقع: المشفر مدرب مسبقًا، أما رأس فئات بيان فيبدأ عشوائيًا.

- GPU: Full fine-tuning قصير.
- CPU: Partial fine-tuning؛ نجمّد معظم المشفر ونحدّث آخر Transformer block مع رأس المهمة.

كلاهما تدريب فعلي، لكن نوعه يسجل في النتائج.

In [ ]:
MODEL_ID = "distilbert/distilbert-base-multilingual-cased"
MAX_LENGTH = 64
BATCH_SIZE = 4
NUM_EPOCHS = 2 if DEVICE.type == "cuda" else 12

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    )
except Exception as exc:
    raise RuntimeError(
        "Checkpoint download failed. Reconnect the runtime and run this cell once. "
        "No API key is required."
    ) from exc

TRAINING_MODE = "full_finetune" if DEVICE.type == "cuda" else "partial_finetune_cpu"
if TRAINING_MODE == "partial_finetune_cpu":
    for parameter in model.base_model.parameters():
        parameter.requires_grad = False
    for parameter in model.base_model.transformer.layer[-1].parameters():
        parameter.requires_grad = True

model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Training mode:", TRAINING_MODE)
print("Epochs:", NUM_EPOCHS)
print(f"Trainable parameters: {trainable:,} / {total:,}")
assert trainable > 0

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training mode: partial_finetune_cpu
Epochs: 12
Trainable parameters: 7,681,540 / 135,327,748


In [ ]:
def iter_batches(examples, batch_size, *, shuffle=False, seed=SEED):
    indexes = np.arange(len(examples))
    if shuffle:
        np.random.default_rng(seed).shuffle(indexes)
    for start in range(0, len(indexes), batch_size):
        yield [examples[index] for index in indexes[start:start + batch_size]]


def encode_batch(batch):
    encoded = tokenizer(
        [row["text"] for row in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(
        [label2id[row["topic"]] for row in batch], dtype=torch.long
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}


def predict(examples):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in iter_batches(examples, BATCH_SIZE):
            encoded = encode_batch(batch)
            logits = model(**encoded).logits
            predictions.extend(logits.argmax(-1).cpu().tolist())
    return [id2label[index] for index in predictions]

print("Batch functions=PASS")

Batch functions=PASS


## 4) خطوة Fine-tuning فعلية

نحدّث أوزانًا داخل آخر Transformer block، ونستخدم validation لاختيار أفضل حقبة، ثم نفتح Test مرة واحدة بعد تثبيت الإعدادات. العينة صغيرة واصطناعية؛ لذا المقارنة دليل تعليمي على سلامة المسار وليست تقديرًا للإنتاج.

# New Section

In [ ]:
learning_rate = 2e-5 if TRAINING_MODE == "full_finetune" else 1e-4
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_parameters, lr=learning_rate)
losses = []
train_steps = 0
epoch_history = []
best_validation_f1 = -1.0
best_epoch = 0
best_trainable_state = None

for epoch_index in range(NUM_EPOCHS):
    model.train()
    epoch_losses = []
    for batch in iter_batches(
        train_rows, BATCH_SIZE, shuffle=True, seed=SEED + epoch_index + 1
    ):
        optimizer.zero_grad(set_to_none=True)
        encoded = encode_batch(batch)
        output = model(**encoded)
        loss = output.loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
        optimizer.step()
        loss_value = float(loss.detach().cpu())
        losses.append(loss_value)
        epoch_losses.append(loss_value)
        train_steps += 1

    epoch_predictions = predict(validation_rows)
    epoch_f1 = f1_score(
        [row["topic"] for row in validation_rows], epoch_predictions,
        labels=LABELS, average="macro", zero_division=0,
    )
    epoch_history.append({
        "epoch": epoch_index + 1,
        "mean_loss": float(np.mean(epoch_losses)),
        "validation_macro_f1": float(epoch_f1),
    })
    if epoch_f1 > best_validation_f1:
        best_validation_f1 = float(epoch_f1)
        best_epoch = epoch_index + 1
        if TRAINING_MODE == "partial_finetune_cpu":
            best_trainable_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter in model.named_parameters()
                if parameter.requires_grad
            }
    print(
        f"epoch={epoch_index + 1:02d} mean_loss={np.mean(epoch_losses):.4f} "
        f"validation_macro_f1={epoch_f1:.4f}"
    )

if best_trainable_state is not None:
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in best_trainable_state:
                parameter.copy_(best_trainable_state[name].to(parameter.device))
    selected_epoch = best_epoch
else:
    selected_epoch = NUM_EPOCHS

assert train_steps >= 1 and all(math.isfinite(value) for value in losses)
assert any(parameter.requires_grad for parameter in model.base_model.parameters())
print("Transformer optimizer steps=PASS", {"steps": train_steps, "selected_epoch": selected_epoch})

epoch=01 mean_loss=1.4031 validation_macro_f1=0.2917
epoch=02 mean_loss=1.3156 validation_macro_f1=0.4167
epoch=03 mean_loss=1.2396 validation_macro_f1=0.4167
epoch=04 mean_loss=1.1272 validation_macro_f1=0.6250
epoch=05 mean_loss=0.9523 validation_macro_f1=0.6250
epoch=06 mean_loss=0.7202 validation_macro_f1=0.7500
epoch=07 mean_loss=0.4770 validation_macro_f1=0.8667
epoch=08 mean_loss=0.2488 validation_macro_f1=0.8667
epoch=09 mean_loss=0.1160 validation_macro_f1=1.0000
epoch=10 mean_loss=0.0552 validation_macro_f1=1.0000
epoch=11 mean_loss=0.0225 validation_macro_f1=1.0000
epoch=12 mean_loss=0.0115 validation_macro_f1=0.8667
Transformer optimizer steps=PASS {'steps': 72, 'selected_epoch': 9}


In [ ]:
transformer_val_pred = predict(validation_rows)
transformer_val_f1 = f1_score(
    [row["topic"] for row in validation_rows],
    transformer_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
validation_delta = transformer_val_f1 - baseline_val_f1
print("Selected epoch:", selected_epoch)
print("Transformer validation macro-F1 (MEASURED_SMOKE):", round(transformer_val_f1, 4))
print("Validation delta vs TF-IDF baseline:", round(validation_delta, 4))
print("Validation predictions:", list(zip(
    [row["topic"] for row in validation_rows], transformer_val_pred
)))
assert 0.0 <= transformer_val_f1 <= 1.0

Selected epoch: 9
Transformer validation macro-F1 (MEASURED_SMOKE): 1.0
Validation delta vs TF-IDF baseline: 0.3333
Validation predictions: [('digital_service', 'digital_service'), ('digital_service', 'digital_service'), ('permit', 'permit'), ('permit', 'permit'), ('health', 'health'), ('health', 'health'), ('transport', 'transport'), ('transport', 'transport')]


In [ ]:
transformer_per_language = {}

for language in sorted({row["language"] for row in validation_rows}):
    indices = [
        index for index, row in enumerate(validation_rows)
        if row["language"] == language
    ]
    language_truth = [
        validation_rows[index]["topic"] for index in indices
    ]
    language_predictions = [
        transformer_val_pred[index] for index in indices
    ]
    transformer_per_language[language] = {
        "examples": len(indices),
        "macro_f1": round(
            f1_score(
                language_truth,
                language_predictions,
                labels=LABELS,
                average="macro",
                zero_division=0,
            ),
            4,
        ),
        "accuracy": round(
            accuracy_score(language_truth, language_predictions),
            4,
        ),
        "errors": sum(
            truth != prediction
            for truth, prediction in zip(
                language_truth, language_predictions
            )
        ),
    }
print(json.dumps(transformer_per_language, ensure_ascii=False, indent=2))

{
  "ar": {
    "examples": 4,
    "macro_f1": 1.0,
    "accuracy": 1.0,
    "errors": 0
  },
  "en": {
    "examples": 4,
    "macro_f1": 1.0,
    "accuracy": 1.0,
    "errors": 0
  }
}


### Transformer Validation Summary

The selected Transformer checkpoint correctly classified all 8 validation
examples, including the two examples misclassified by the TF-IDF baseline.
It achieved a validation Macro-F1 of 1.0000 with 0 validation errors.

Arabic and English each achieved a Macro-F1 of 1.0000 with 0 errors.
These results are descriptive because the validation set is small and do
not represent production performance.

## 5) القياس النهائي المصغر

بعد تثبيت الإعدادات السابقة نقيس Test مرة واحدة. لا نغير الإعدادات بناء على هذه النتيجة ثم نعيد تسميتها Frozen Test.

In [ ]:
test_truth = [row["topic"] for row in test_rows]
baseline_test_pred = baseline.predict([row["text"] for row in test_rows]).tolist()
transformer_test_pred = predict(test_rows)

results = {
    "result_type": "MEASURED_SMOKE",
    "data_source": DATA_SOURCE,
    "model_id": MODEL_ID,
    "device": str(DEVICE),
    "training_mode": TRAINING_MODE,
    "seed": SEED,
    "epochs_run": NUM_EPOCHS,
    "selected_epoch": selected_epoch,
    "train_steps": train_steps,
    "mean_train_loss": float(np.mean(losses)),
    "baseline_validation_macro_f1": float(baseline_val_f1),
    "transformer_validation_macro_f1": float(transformer_val_f1),
    "validation_delta_vs_baseline": float(validation_delta),
    "baseline_beaten_on_validation": bool(validation_delta > 0),
    "baseline_test_macro_f1": float(f1_score(
        test_truth, baseline_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_macro_f1": float(f1_score(
        test_truth, transformer_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_accuracy": float(accuracy_score(test_truth, transformer_test_pred)),
    "limitations": [
        "synthetic tiny dataset",
        "small validation set used for epoch selection",
        "not an estimate of production quality",
    ],
}
print(json.dumps(results, ensure_ascii=False, indent=2))
Path("day2_classification_metrics.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
)

{
  "result_type": "MEASURED_SMOKE",
  "data_source": "github_course_file",
  "model_id": "distilbert/distilbert-base-multilingual-cased",
  "device": "cpu",
  "training_mode": "partial_finetune_cpu",
  "seed": 42,
  "epochs_run": 12,
  "selected_epoch": 9,
  "train_steps": 72,
  "mean_train_loss": 0.6407361574496867,
  "baseline_validation_macro_f1": 0.6666666666666666,
  "transformer_validation_macro_f1": 1.0,
  "validation_delta_vs_baseline": 0.33333333333333337,
  "baseline_beaten_on_validation": true,
  "baseline_test_macro_f1": 0.7333333333333333,
  "transformer_test_macro_f1": 0.8666666666666667,
  "transformer_test_accuracy": 0.875,
  "limitations": [
    "synthetic tiny dataset",
    "small validation set used for epoch selection",
    "not an estimate of production quality"
  ]
}


800

In [ ]:
test_per_language = {}

for language in sorted({row["language"] for row in test_rows}):
    indices = [
        index for index, row in enumerate(test_rows)
        if row["language"] == language
    ]

    language_truth = [test_truth[index] for index in indices]
    baseline_predictions = [
        baseline_test_pred[index] for index in indices
    ]
    transformer_predictions = [
        transformer_test_pred[index] for index in indices
    ]

    test_per_language[language] = {
        "examples": len(indices),
        "baseline": {
            "macro_f1": round(
                f1_score(
                    language_truth,
                    baseline_predictions,
                    labels=LABELS,
                    average="macro",
                    zero_division=0,
                ),
                4,
            ),
            "accuracy": round(
                accuracy_score(language_truth, baseline_predictions),
                4,
            ),
            "errors": sum(
                truth != prediction
                for truth, prediction in zip(
                    language_truth, baseline_predictions
                )
            ),
        },
        "transformer": {
            "macro_f1": round(
                f1_score(
                    language_truth,
                    transformer_predictions,
                    labels=LABELS,
                    average="macro",
                    zero_division=0,
                ),
                4,
            ),
            "accuracy": round(
                accuracy_score(language_truth, transformer_predictions),
                4,
            ),
            "errors": sum(
                truth != prediction
                for truth, prediction in zip(
                    language_truth, transformer_predictions
                )
            ),
        },
    }

transformer_test_errors = [
    {
        "example_id": row["example_id"],
        "language": row["language"],
        "text": row["text"],
        "true_topic": row["topic"],
        "predicted_topic": prediction,
    }
    for row, prediction in zip(test_rows, transformer_test_pred)
    if row["topic"] != prediction
]
print("Frozen test results by language:")
print(json.dumps(test_per_language, ensure_ascii=False, indent=2))
print("\nTransformer frozen-test errors:")
print(json.dumps(transformer_test_errors, ensure_ascii=False, indent=2))

assert len(transformer_test_errors) == 1

Frozen test results by language:
{
  "ar": {
    "examples": 4,
    "baseline": {
      "macro_f1": 1.0,
      "accuracy": 1.0,
      "errors": 0
    },
    "transformer": {
      "macro_f1": 0.6667,
      "accuracy": 0.75,
      "errors": 1
    }
  },
  "en": {
    "examples": 4,
    "baseline": {
      "macro_f1": 0.3333,
      "accuracy": 0.5,
      "errors": 2
    },
    "transformer": {
      "macro_f1": 1.0,
      "accuracy": 1.0,
      "errors": 0
    }
  }
}

Transformer frozen-test errors:
[
  {
    "example_id": "D-009",
    "language": "ar",
    "text": "الخدمة الإلكترونية واضحة وسريعة",
    "true_topic": "digital_service",
    "predicted_topic": "health"
  }
]


### Frozen Test Slice and Error Analysis

The frozen test was evaluated once after all model-selection decisions were
fixed. Across the 8 test examples, the Transformer achieved a Macro-F1 of
0.8667 and an accuracy of 0.875, compared with a TF-IDF Macro-F1 of 0.7333.

On the Arabic slice, TF-IDF correctly classified all 4 examples, while the
Transformer correctly classified 3 of 4. Its only frozen-test error was the
Arabic example D-009, where a `digital_service` text was predicted as
`health`.

On the English slice, the Transformer correctly classified all 4 examples,
while TF-IDF correctly classified 2 of 4. Therefore, the Transformer's
overall improvement was mainly driven by stronger English performance.
These slice results are descriptive because each language contains only
four test examples. No model tuning was performed after viewing the frozen
test.

لا تستنتج أن Transformer «فشل» أو «نجح إنتاجيًا» من هذه العينة. سجل الفرق، ثم اكتب ما تحتاجه لتقدير موثوق: بيانات أكثر، عدة بذور، slices، وتقييم مجمد.

## مستويات التحدي

- **Core:** الخلايا السابقة فقط.
- **Explore:** على GPU قارن full fine-tuning بالمسار الجزئي، مع تثبيت البيانات والبذرة والـvalidation.
- **Distinction:** نفذ ثلاث بذور وسجل mean ± range؛ لا تستخدم Test لاختيار البذرة.

In [ ]:
core_checks = {
    "split_isolation": split_report["group_overlap"] == 0,
    "baseline_valid": 0.0 <= baseline_val_f1 <= 1.0,
    "training_ran": train_steps >= 1,
    "transformer_weights_updated": any(
        parameter.requires_grad for parameter in model.base_model.parameters()
    ),
    "loss_is_finite": all(math.isfinite(value) for value in losses),
    "validation_valid": 0.0 <= transformer_val_f1 <= 1.0,
    "baseline_comparison_recorded": math.isfinite(validation_delta),
    "result_is_honest": results["result_type"] == "MEASURED_SMOKE",
    "bilingual_data": {"ar", "en"} <= {row["language"] for row in rows},
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY2_NOTEBOOK3_CORE=PASS")

split_isolation: PASS
baseline_valid: PASS
training_ran: PASS
transformer_weights_updated: PASS
loss_is_finite: PASS
validation_valid: PASS
baseline_comparison_recorded: PASS
result_is_honest: PASS
bilingual_data: PASS
DAY2_NOTEBOOK3_CORE=PASS


## نقطة GitHub

1. احفظ نسخة notebook في مستودعك بالاسم نفسه.
2. لا ترفع Hugging Face cache أو model weights.
3. ارفع `day2_classification_metrics.json` ضمن `reports/` إذا لم يحتو بيانات شخصية.
4. حدّث `DECISIONS.md` بنوع التدريب والسبب وحدود العينة.
5. لا تعمل commit النهائي لليوم حتى تكمل دفتر NER وQA.

**التالي:** [مختبر NER وQA](04_ner_and_qa.ipynb).